Análise Exploratória de Dados (AED)

Objetivo

Este notebook tem como objetivo realizar a análise exploratória dos dados de pedidos armazenados no Azure Data Lake Storage Gen2 (ADLS Gen2), compreendendo a estrutura dos dados, avaliando sua qualidade e identificando possíveis oportunidades de análise.

1. Instalação das Bibliotecas

In [0]:
%pip install azure-storage-file-datalake azure-identity pandas pyarrow python-dotenv

dbutils.library.restartPython()

2. Importação das Bibliotecas

Nesta etapa importamos as bibliotecas necessárias para autenticação, acesso ao Azure Data Lake e manipulação dos dados.

In [0]:
from dotenv import load_dotenv 
import os 

from azure.identity import ClientSecretCredential 
from azure.storage.filedatalake import DataLakeServiceClient 

import pandas as pd 
from io import BytesIO

3. Carregamento das Credenciais

As credenciais são carregadas a partir do arquivo .env, evitando a exposição de informações sensíveis no código.

In [0]:
caminho_env = "/Workspace/Users/jeronimo.carlos104@gmail.com/estagio-empregadados-turma-2/.env"

load_dotenv(caminho_env, override=True) 

client_id = os.getenv("ADLS_CLIENT_ID") 
tenant_id = os.getenv("ADLS_TENANT_ID") 
client_secret = os.getenv("ADLS_CLIENT_SECRET") 

print("Client ID carregado:", client_id is not None) 
print("Tenant ID carregado:", tenant_id is not None) 
print("Client Secret carregado:", client_secret is not None)

4. Conexão com o Azure Data Lake

Criamos a conexão com o Storage Account e acessamos o container raw.


In [0]:
storage_account_name = "internshipdatalake" 

credential = ClientSecretCredential( 
    tenant_id, 
    client_id, 
    client_secret 
) 

service_client = DataLakeServiceClient( 
    account_url=f"https://{storage_account_name}.dfs.core.windows.net", 
    credential=credential, 
) 

fs_client = service_client.get_file_system_client("raw") 

print("Conexão realizada com sucesso.")

5. Localização da Pasta Mais Recente

Como os arquivos são armazenados em diretórios organizados por data e horário, buscamos automaticamente o diretório mais recente

In [0]:
def encontrar_pasta_mais_recente(fs_client, base_path="real-time-data"):
    pastas = set()

    for p in fs_client.get_paths(path=base_path, recursive=True):
        if not p.is_directory:
            pasta = "/".join(p.name.split("/")[:-1])
            pastas.add(pasta)

    return sorted(pastas)[-1]


pasta_mais_recente = encontrar_pasta_mais_recente(fs_client)

print("Pasta selecionada:")
print(pasta_mais_recente)

6. Leitura do Arquivo Parquet

Nesta etapa realizamos a leitura do arquivo de pedidos.

In [0]:
def ler_parquet_do_adls(caminho_no_container):
    file_client = fs_client.get_file_client(caminho_no_container)

    download = file_client.download_file()

    conteudo = download.readall()

    return pd.read_parquet(BytesIO(conteudo))
df_pedidos_pd = ler_parquet_do_adls(
    f"{pasta_mais_recente}/ecommerce_pedidos.parquet"
)

df_pedidos = spark.createDataFrame(df_pedidos_pd)

print("Quantidade de registros:", df_pedidos.count())

7. Visualização Inicial dos Dados

A primeira inspeção permite compreender como os dados estão organizados.

In [0]:
display(df_pedidos.limit(10))

8. Estrutura do Dataset

Analisamos os nomes das colunas e seus respectivos tipos de dados.

In [0]:
df_pedidos.printSchema()

9. Quantidade de Linhas e Colunas

In [0]:
print(f"Quantidade de registros: {df_pedidos.count()}") 
print(f"Quantidade de colunas: {len(df_pedidos.columns)}")

10. Estatísticas Descritivas

Apresentamos medidas estatísticas básicas para colunas numéricas.

In [0]:
display(df_pedidos.describe())

11. Verificação de Valores Nulos

A identificação de valores ausentes é uma etapa importante para avaliar a qualidade dos dados.

In [0]:
from pyspark.sql.functions import col 
from pyspark.sql.functions import count 
from pyspark.sql.functions import when 
display
( df_pedidos.select([ 
                     count( 
                           when(col(c).isNull(), c) 
                           ).alias(c) 
                     for c in df_pedidos.columns
                ]) 
 )

12. Verificação de Registros Duplicados

In [0]:
total_registros = df_pedidos.count()

registros_unicos = df_pedidos.dropDuplicates().count() 

print(f"Total de registros: {total_registros}") 

print(f"Registros únicos: {registros_unicos}") 

print(f"Duplicados: {total_registros - registros_unicos}")